In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

import xgboost as xgb
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, roc_auc_score
import os

In [2]:
# Setup spark
spark = ( 
    SparkSession.builder 
    .master("local[*]")
    .appName("Spark RAG Maintenance")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/06 17:29:30 WARN Utils: Your hostname, MSI, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/06 17:29:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/06 17:29:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1


In [3]:
# Load data
COL_NAMES = [
    "engine_id",
    "cycle",
    "op_setting_1",
    "op_setting_2",
    "op_setting_3",
    *[f"sensor_{i}" for i in range(1, 22)]
]

def load_cmaps(path: str):
    df = spark.read.csv(path, sep=" ", inferSchema=True)
    df = df.drop("_c26", "_c27")  # Drop empty columns
    return df.toDF(*COL_NAMES)  # Rename columns

train_df = load_cmaps("../data/raw/train_FD001.txt")
test_df = load_cmaps("../data/raw/test_FD001.txt") 

print(f"Train rows: {train_df.count():,}")
print(f"Test rows: {test_df.count():,}")
train_df.show(3)

Train rows: 20,631
Test rows: 13,096


26/03/06 17:29:40 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------+-----+------------+------------+------------+--------+--------+--------+--------+--------+--------+--------+--------+--------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+
|engine_id|cycle|op_setting_1|op_setting_2|op_setting_3|sensor_1|sensor_2|sensor_3|sensor_4|sensor_5|sensor_6|sensor_7|sensor_8|sensor_9|sensor_10|sensor_11|sensor_12|sensor_13|sensor_14|sensor_15|sensor_16|sensor_17|sensor_18|sensor_19|sensor_20|sensor_21|
+---------+-----+------------+------------+------------+--------+--------+--------+--------+--------+--------+--------+--------+--------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+
|        1|    1|     -7.0E-4|     -4.0E-4|       100.0|  518.67|  641.82|  1589.7|  1400.6|   14.62|   21.61|  554.36| 2388.06| 9046.19|      1.3|    47.47|   521.66|  2388.02|  8138.62|   8.4195|     0.03|      392|     2388

In [4]:
max_cycles = train_df.groupBy("engine_id").agg(
        F.max("cycle").alias("max_cycle")
)

train_df = train_df.join(max_cycles, on="engine_id")
train_df = train_df.withColumn("RUL", F.col("max_cycle") - F.col("cycle"))
train_df = train_df.withColumn("failure_soon", (F.col("RUL") <= 30).cast("int"))

print("Label distribution (failure soon):")
train_df.groupBy("failure_soon").count().show()

Label distribution (failure soon):
+------------+-----+
|failure_soon|count|
+------------+-----+
|           1| 3100|
|           0|17531|
+------------+-----+



In [5]:
SENSOR_COLS = [f"sensor_{i}" for i in range(1, 22)]

# 5 cycle rolling window per engine
w5 = Window.partitionBy("engine_id").orderBy("cycle").rowsBetween(-4, 0)

# 1 cycle lag window
w_lag = Window.partitionBy("engine_id").orderBy("cycle")

# For each col add rolling mean, stddev and 1 cycle lag as new features
for col in SENSOR_COLS:
    train_df = (
        train_df
        .withColumn(f"{col}_roll_mean", F.mean(col).over(w5))
        .withColumn(f"{col}_std_std",   F.stddev(col).over(w5))
        .withColumn(f"{col}_lag_1",     F.lag(col, 1).over(w_lag))
    )

# Drop rows with nulls
train_df = train_df.na.drop()

print("After feature engineering:")
print(f"Rows: {train_df.count():,}")
print(f"Columns: {len(train_df.columns)}")
train_df.printSchema()

After feature engineering:


Rows: 20,531
Columns: 92
root
 |-- engine_id: integer (nullable = true)
 |-- cycle: integer (nullable = true)
 |-- op_setting_1: double (nullable = true)
 |-- op_setting_2: double (nullable = true)
 |-- op_setting_3: double (nullable = true)
 |-- sensor_1: double (nullable = true)
 |-- sensor_2: double (nullable = true)
 |-- sensor_3: double (nullable = true)
 |-- sensor_4: double (nullable = true)
 |-- sensor_5: double (nullable = true)
 |-- sensor_6: double (nullable = true)
 |-- sensor_7: double (nullable = true)
 |-- sensor_8: double (nullable = true)
 |-- sensor_9: double (nullable = true)
 |-- sensor_10: double (nullable = true)
 |-- sensor_11: double (nullable = true)
 |-- sensor_12: double (nullable = true)
 |-- sensor_13: double (nullable = true)
 |-- sensor_14: double (nullable = true)
 |-- sensor_15: double (nullable = true)
 |-- sensor_16: double (nullable = true)
 |-- sensor_17: integer (nullable = true)
 |-- sensor_18: integer (nullable = true)
 |-- sensor_19: double (nul

In [6]:
# Store as parquet
os.makedirs("../data/processed", exist_ok=True)
train_df.write.mode("overwrite").parquet("../data/processed/train.parquet")
print("Saved processed data to ../data/processed/train.parquet")

verify = spark.read.parquet("../data/processed/train.parquet")
print("Parquet read-back:") 
print(f"Rows: {verify.count():,}")
print(f"Colrs: {len(verify.columns)}")
verify.printSchema()

Saved processed data to ../data/processed/train.parquet
Parquet read-back:
Rows: 20,531
Colrs: 92
root
 |-- engine_id: integer (nullable = true)
 |-- cycle: integer (nullable = true)
 |-- op_setting_1: double (nullable = true)
 |-- op_setting_2: double (nullable = true)
 |-- op_setting_3: double (nullable = true)
 |-- sensor_1: double (nullable = true)
 |-- sensor_2: double (nullable = true)
 |-- sensor_3: double (nullable = true)
 |-- sensor_4: double (nullable = true)
 |-- sensor_5: double (nullable = true)
 |-- sensor_6: double (nullable = true)
 |-- sensor_7: double (nullable = true)
 |-- sensor_8: double (nullable = true)
 |-- sensor_9: double (nullable = true)
 |-- sensor_10: double (nullable = true)
 |-- sensor_11: double (nullable = true)
 |-- sensor_12: double (nullable = true)
 |-- sensor_13: double (nullable = true)
 |-- sensor_14: double (nullable = true)
 |-- sensor_15: double (nullable = true)
 |-- sensor_16: double (nullable = true)
 |-- sensor_17: integer (nullable = tr

In [7]:
# Conver to pandas
pdf = spark.read.parquet("../data/processed/train.parquet").toPandas()
EXCLUDE_COLS = ["engine_id", "cycle", "max_cycle", "RUL", "failure_soon"]
FEATURE_COLS = [col for col in pdf.columns if col not in EXCLUDE_COLS]

X = pdf[FEATURE_COLS]
y_rul = pdf["RUL"].values
y_cls = pdf["failure_soon"].values

X_train, X_val, y_rul_tr, y_rul_val, y_cls_tr, y_cls_val = train_test_split(
    X, y_rul, y_cls, test_size=0.2, random_state=42
)

print(f"Train shape: {X_train.shape}")
print(f"Validation shape: {X_val.shape}")
print(f"Features used: {len(FEATURE_COLS)}")

Train shape: (16424, 87)
Validation shape: (4107, 87)
Features used: 87


In [8]:
# Regression model for RUL
reg_model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.6,
    subasmple=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

reg_model.fit(
    X_train, y_rul_tr,
    eval_set=[(X_val, y_rul_val)],
    verbose=50
)

rul_preds = reg_model.predict(X_val)
rul_rmse = np.sqrt(mean_squared_error(y_rul_val, rul_preds))
print(f"RUL Regression RMSE: {rul_rmse:.4f}")

[0]	validation_0-rmse:46.00314


/home/rm_a/dev/spark-rag-maintenance/.venv/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [17:47:35] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "subasmple" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[50]	validation_0-rmse:39.98423
[100]	validation_0-rmse:40.69619
[150]	validation_0-rmse:40.85907
[200]	validation_0-rmse:40.93655
[250]	validation_0-rmse:40.93176
[299]	validation_0-rmse:40.95710
RUL Regression RMSE: 40.9571


In [9]:
# Classification model for failure soon
cls_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    scale_pos_weight=(y_cls_tr == 0).sum() / (y_cls_tr == 1).sum(), # Handle class imbalance
    random_state=42,
    n_jobs=-1
)

cls_model.fit(
    X_train, y_cls_tr,
    eval_set=[(X_val, y_cls_val)],
    verbose=50
)

cls_probs = cls_model.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_cls_val, cls_probs)
print("Failure Classification ROC-AUC: {auc:.4f}")


[0]	validation_0-logloss:0.65130
[50]	validation_0-logloss:0.13800
[100]	validation_0-logloss:0.10475
[150]	validation_0-logloss:0.09443
[200]	validation_0-logloss:0.08647
[250]	validation_0-logloss:0.08080
[299]	validation_0-logloss:0.07628
Failure Classification ROC-AUC: {auc:.4f}


In [10]:
# Save models
os.makedirs("../models", exist_ok=True)
reg_model.save_model("../models/xgb_rul.json")
cls_model.save_model("../models/xgb_failure_cls.json")
print("Models saved to ../models/")

Models saved to ../models/
